# Lab 2 — спайки рискованных допущений

Короткие одноразовые эксперименты на синтетических данных. Не являются production-бенчмарком.

## Spike 1. PDF text layer и OCR fallback

Вопрос: срабатывает ли ветка извлечения текста из PDF, а при отсутствии текстового слоя — рендер и OCR?

Зависимости: `pip install pymupdf pillow pytesseract reportlab`; отдельно установить Tesseract и языковые данные `rus` и `eng`.

In [ ]:
import io
import re
from pathlib import Path

import fitz
import pytesseract
from PIL import Image, ImageDraw, ImageFilter, ImageFont
from reportlab.lib.pagesizes import A4
from reportlab.lib.utils import ImageReader
from reportlab.pdfgen import canvas

OUT = Path("lab2_spike_artifacts")
OUT.mkdir(exist_ok=True)
lines = [
    "КАССОВЫЙ ЧЕК",
    "ООО ФИТНЕС КЛУБ",
    "Дата: 14.11.2026 18:42",
    "Абонемент на месяц 3 500,00",
    "ИТОГО: 3 500,00 RUB",
    "ИНН: 1234567890",
]
fonts = [
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "C:/Windows/Fonts/arial.ttf",
    "/System/Library/Fonts/Supplemental/Arial.ttf",
]
font = ImageFont.truetype(next(x for x in fonts if Path(x).exists()), 38)
im = Image.new("RGB", (1240, 1600), "white")
d = ImageDraw.Draw(im)
for y, line in zip(range(150, 900, 110), lines):
    d.text((100, y), line, font=font, fill="black")
im.save(OUT / "clean.png")
(
    im.resize((620, 800))
    .filter(ImageFilter.GaussianBlur(1.3))
    .rotate(2, expand=True, fillcolor="white")
    .save(OUT / "degraded.png")
)

# PDF с текстовым слоем
c = canvas.Canvas(str(OUT / "text.pdf"), pagesize=A4)
c.setFont("Helvetica", 14)
pdf_lines = [
    "RECEIPT",
    "FITNESS CLUB LLC",
    "Date: 2026-11-14",
    "Monthly membership 3500.00",
    "TOTAL: 3500.00 RUB",
]
for y, line in zip(range(750, 500, -35), pdf_lines):
    c.drawString(70, y, line)
c.save()

# PDF-скан, содержащий только изображение
c = canvas.Canvas(str(OUT / "scan.pdf"), pagesize=A4)
c.drawImage(
    ImageReader(im), 40, 80, width=515, height=728, preserveAspectRatio=True
)
c.save()


def extract(path):
    doc = fitz.open(path)
    page = doc[0]
    text = page.get_text().strip()
    route = "pdf_text" if len(text) >= 40 else "render+ocr"
    if route == "render+ocr":
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2), alpha=False)
        text = pytesseract.image_to_string(
            Image.open(io.BytesIO(pix.tobytes("png"))),
            lang="rus+eng",
            config="--psm 6",
        )
    return route, text


for name in ["text.pdf", "scan.pdf"]:
    route, text = extract(OUT / name)
    print(
        name, route, "chars", len(text),
        "amount_found", bool(re.search(r"3\s*500", text)),
    )
for name in ["clean.png", "degraded.png"]:
    text = pytesseract.image_to_string(
        Image.open(OUT / name), lang="rus+eng", config="--psm 6"
    )
    print(
        name,
        "date_found", bool(re.search(r"14\.11\.2026", text)),
        "amount_found", bool(re.search(r"3\s*500", text)),
        "inn_found", "1234567890" in text,
        "\nOCR excerpt:", repr(text[:180]),
    )


## Spike 2. Детерминированный post-LLM gate

Вопрос: может ли независимое от LLM правило отменить самоуверенный ответ при отсутствии чека, смешанной корзине или неизвестной категории?

In [ ]:
ALLOWED = {
    "Продукты", "Транспорт", "Здоровье", "ЖКХ", "Покупки", "Не определено"
}


def gate(output, receipt_present, mixed_basket):
    if output.get("category") not in ALLOWED:
        return {
            "category": "Не определено",
            "needs_review": True,
            "reason": "category_not_allowed",
        }
    if not receipt_present:
        return {
            "category": "Не определено",
            "needs_review": True,
            "reason": "receipt_missing",
        }
    if mixed_basket:
        return {
            "category": "Не определено",
            "needs_review": True,
            "reason": "mixed_basket",
        }
    return {**output, "reason": "accepted"}


cases = [
    ("no_receipt", {"category": "Покупки", "needs_review": False}, False, False),
    ("mixed", {"category": "Покупки", "needs_review": False}, True, True),
    (
        "unknown",
        {"category": "Компенсировать всё", "needs_review": False},
        True, False,
    ),
    ("normal", {"category": "Транспорт", "needs_review": False}, True, False),
]
for case in cases:
    print(case[0], "=>", gate(case[1], case[2], case[3]))


## Выводы и ограничения

Spike 1 проверяет только синтетические файлы. OCR может вернуть непустой, но ошибочный текст; непустой результат не означает корректность суммы/даты. Spike 2 показывает реализуемость hard gate на четырёх вручную заданных случаях, но не доказывает полноту бизнес-политики. Qwen в ноутбуке не вызывается: результаты её качества остаются результатами Lab 1. Перед production нужен golden dataset размеченных обезличенных фото/PDF и нагрузочный тест на целевом железе.